<a id="top"></a>

# Demo: a tool call is just tokens

```yaml
title: "Demo: a tool call is just tokens"
keywords: tool call, tool_calls, bind_tools, tool definition, schema, reasoning is tokens, protocol, langchain, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Teacher demo — Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: dry-run before class and confirm the model reaches for the tool at least most of the time. Clear outputs before committing.
>
> **Model-agnostic by design.** From L03 on the course drives the model through a **LangChain chat model** (`ChatAnthropic` here) instead of a raw vendor SDK — swap it for `ChatOpenAI` and the same tool code runs unchanged. The tool is a plain typed function; `bind_tools` derives its schema from the signature and docstring. The key still loads through the config seam (`require_anthropic_key`); we never hard-code it.
>
> The point to land: a tool call is **tokens the model emitted**, not an action it took. We stop the moment `.tool_calls` arrives and inspect it — we do **not** run the calculator.

## Contents

- [1. Setup](#1-setup)
- [2. No tools bound — the L01–L06 world](#2-no-tools-bound--the-l01l06-world)
- [3. Bind the tool — and stop at the call](#3-bind-the-tool--and-stop-at-the-call)
- [4. What just happened (and didn't)](#4-what-just-happened-and-didnt)
- [5. Takeaways](#5-takeaways)

## 1. Setup

A LangChain `ChatAnthropic` model, the single `calculator` tool (a plain typed function), and the tool bound to the model with `bind_tools`. Anchor model: **Claude Sonnet 4.6**. Live cells are gated on `LIVE` so a keyless run still executes top to bottom.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage

from fluffy_potato_curriculum.common.config import get_settings, require_anthropic_key

SONNET = "claude-sonnet-4-6"

# Live-by-default: build the model only when a key is present, so a keyless
# restart-and-run-all still passes (the live cells below are gated on LIVE).
LIVE = bool(get_settings().anthropic_api_key)


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the exact result as a string.

    Use this whenever the user asks for the result of a calculation. Restricted to
    digits and the operators + - * / ( ) . and whitespace so a hallucinated
    expression cannot run arbitrary code. Raises ValueError on anything else — that
    rejection is the whole point of Demo 4.
    """
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# The tool definition is DERIVED from the function above — its name, its docstring
# (the description), and its typed signature (the argument schema). We never
# hand-write a JSON schema; the typed function *is* the source of it.
TOOLS = [calculator]

# A prompt the model is unreliable at without a calculator, so it has a real
# reason to reach for the tool.
PROMPT = "What is 18,374 multiplied by 92,431?"

if LIVE:
    # Constructing the model reads the key through common.config, never a raw
    # os.environ lookup. Swap ChatAnthropic -> ChatOpenAI and nothing else changes.
    model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)
    bound = model.bind_tools(TOOLS)  # hands the derived definition to the model

print("setup ready (LIVE =", LIVE, ")")

**The definition the model actually sees.** You wrote a plain function, but the model still receives a structured tool definition — LangChain derived it from the function's name, docstring, and type hints. Print it once so "the schema still exists" is concrete.

In [ ]:
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

# The tool DEFINITION still exists — you just didn't hand-write it. This is exactly
# what bind_tools sends to the model: a name, a description (from the docstring), and
# a JSON-Schema for the arguments (from the type hints). No key needed to see it.
print(json.dumps(convert_to_openai_tool(calculator), indent=2))

[↑ Back to top](#top)

## 2. No tools bound — the L01–L06 world

First, the exact prompt with **no tools**. The model answers directly, often with a confident, wrong number. This is text-in, text-out — everything L01–L06 did.

In [ ]:
if LIVE:
    # The plain model, no tools bound. It answers directly -- text in, text out,
    # everything L01-L06 did -- often with a confident, wrong number.
    no_tools = model.invoke([HumanMessage(PROMPT)])
    print("text ->", no_tools.text)
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 3. Bind the tool — and stop at the call

Now use `bound` (the model with the tool bound) and the **same** prompt. We stop as soon as the reply arrives — we do **not** run the calculator. Print `.tool_calls` (the parsed requests) and `.text` (usually empty on a tool turn) so both are visible.

In [ ]:
if LIVE:
    resp = bound.invoke([HumanMessage(PROMPT)])
    print("text on this reply :", repr(resp.text))  # usually '' on a tool turn
    print("tool calls proposed:", len(resp.tool_calls))
    for call in resp.tool_calls:
        # each call is a plain dict: {"name", "args", "id"}
        print(f"  tool call  name={call['name']!r}  id={call['id']!r}  args={call['args']!r}")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 4. What just happened (and didn't)

Read the tool call aloud: a tool **name** (`calculator`), an **arguments** dict (`{'expression': ...}`), and a unique **id**. Nothing was computed — the model did not multiply anything. It emitted *tokens that say* 'I would like the calculator run with this expression,' and LangChain parsed them into this dict. The number inside the arguments came from the model's tokens, not from arithmetic.

In [ ]:
if LIVE:
    call = resp.tool_calls[0]
    print("the model PROPOSED this call (it did not run it):")
    print("  name      :", call["name"])
    print("  arguments :", call["args"])
    print("  call id   :", call["id"])
    print("\nThree actors, only one has acted: the MODEL proposed; the APPLICATION (us)")
    print("holds the call and has done nothing; the TOOL (calculator) has not run.")
else:
    print("offline: set ANTHROPIC_API_KEY to run this live cell")

[↑ Back to top](#top)

## 5. Takeaways

- A tool call is **just more structured tokens** the model emitted — LangChain parsed them into `.tool_calls` for you, but nothing *ran*. Same shape-contract idea as L06's `<thinking>`/`<answer>` tags, except now the application is expected to *act* on the shape.
- The tool **definition still exists** — you saw it printed — you just didn't hand-write it: `bind_tools` derived it from the typed function and its docstring.
- **Three actors**, only one has moved: model proposed, application holds the call, tool has not run. The rest of the lesson fills in the remaining steps.
- The decision to emit a tool call *instead of* answering directly is itself a **reasoning step** (reinforce L06 — don't re-teach). A vague tool is one the model skips ([L08](L0702_lecture.md) answers *why*).
- Next: [Demo 2](L0704_lecture.ipynb) completes the loop this demo stopped halfway through.

[↑ Back to top](#top)